# 1 : Importing data

In [1]:
from pathlib import Path

content=[]
for path in Path("raw data").rglob("*.md"):
    text=path.read_text(encoding="utf-8")
    content.append(text)

print(len(content))

20


# 2 : Preparing data

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

data=[]

for c in content:
    for piece in splitter.split_text(c):
        data.append(piece)

print(len(data))

208


# 3 : Embedding data

In [3]:
import ollama
import numpy as np

def embed(texts,batch_size=32):
    vectors=[]
    for i in range(0,len(texts),batch_size):
        response=ollama.embed(
            model="nomic-embed-text",
            input=texts[i:i+batch_size],
        )
        vectors.extend(response["embeddings"])

    return np.array(vectors,dtype="float32")

In [4]:
embeddings=embed(data)

# 4 : Indexing data

In [5]:
from faiss import normalize_L2,IndexFlatIP

normalize_L2(embeddings)
index=IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

#print(index.ntotal)

In [6]:
def retrieve(query,k):
    embedded_query=embed([query])
    normalize_L2(embedded_query)
    D, I =index.search(embedded_query,k)
    return [data[i] for i in I[0]]

# 5 : Generating response with code

In [7]:
def generate(prompt):

    response=ollama.chat(
        model="llama3.2",
        messages=[
            {
                'role': 'user',
                'content': prompt,
            },
        ],
        options={"temperature":0}
    )

    return response['message']['content']

In [8]:
def generate_answer_no_rag(query):
    prompt=f"""
    You are a helpful assistant.

    The question is about Docker.

    Question :
    {query}

    Rules : 
    -Answer in one sentence.

    """

    return generate(prompt)

In [9]:
memory=[]
def generate_answer(query):

    context="\n".join(retrieve(query,3))

    history=""
    for q,a in memory[-3:]:
        history+=f"User : {q} \nAssistant : {a}\n"

    prompt=f"""
    You are a helpful assistant.

    Rules : 
    -when the user asks something that doesn't exist in the data you have, say exactly "I don't have enough data".
    - If the context contains no relevant information, say exactly "I don't have enough data".
    - If you don't have an answer, say you don't have enough data.
    - Be concise and friendly.
    - Answer in one sentence.

    Conversation history:
    {history}

    Context from knowledge base:
    {context}

    Question :
    {query}
    """

    answer= generate(prompt)
    memory.append((query,answer))
    return answer



# 6 : Generating response with n8n

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

def get_env(name):
    value=os.environ.get(name)
    if value is None:
        raise RuntimeError(f"Missing variable : {name}")

    return value

post_url=get_env("post_url")
query_url=get_env("query_url")

In [11]:
import requests

data_loaded=False

def trigger_data_loading_n8n(post_url):
    trigger=requests.post(post_url)
    trigger.raise_for_status()
    return True


data_loaded=trigger_data_loading_n8n(post_url)

print("data loaded ? : ",data_loaded)

data loaded ? :  True


In [12]:
def ask_n8n(query_url,query):

    question=requests.post(
        query_url,
        json={
            "question":query,
            "session_id": session_id}
    )

    question.raise_for_status()
    response=question.json().get("output")
    return(response)



# 7 : Loading data into csv file

In [13]:
import uuid
import pandas as pd

def load_results(data_file):
    global memory
    global session_id

    memory= []
    bilan=[]
    session_id=str(uuid.uuid4())

    qa=pd.read_csv(data_file,sep=";")
    
    
    for i, row in qa.iterrows():
        result={}
        result["question_nb"]=i+1
        question=row["question"]
        result["question"]=question
        result["expected_answer"]=row["expected"]

        result["no_rag_code_answer"]=generate_answer_no_rag(question)

        result["code_answer"]=generate_answer(question)

        result["n8n_answer"]=ask_n8n(query_url,question)

        bilan.append(result)

        print("results : ",result)
    return bilan

     

In [14]:
bilan=load_results("q&a/q&a.csv")

results :  {'question_nb': 1, 'question': 'What is Docker?', 'expected_answer': 'Docker is an open platform for developing, shipping, and running applications.', 'no_rag_code_answer': 'Docker is a containerization platform that allows developers to package, ship, and run applications in containers, which are lightweight and standalone, providing a consistent and reliable way to deploy applications across different environments.', 'code_answer': 'Docker is an open platform for developing, shipping, and running applications.', 'n8n_answer': 'Docker is an open platform for developing, shipping, and running applications, allowing you to separate your applications from your infrastructure and deliver software quickly.'}
results :  {'question_nb': 2, 'question': 'What is a container image?', 'expected_answer': 'A container image is a standardized package that includes all of the files, binaries, libraries, and configurations to run a container.', 'no_rag_code_answer': 'A container image is a

In [15]:
bilan_df=pd.DataFrame.from_dict(bilan)
bilan_df.head(3)

,question_nb,question,expected_answer,no_rag_code_answer,code_answer,n8n_answer
0,1,What is Docker?,"Docker is an open platform for developing, shi...",Docker is a containerization platform that all...,"Docker is an open platform for developing, shi...","Docker is an open platform for developing, shi..."
1,2,What is a container image?,A container image is a standardized package th...,"A container image is a lightweight, packaged r...",A container image is a standardized package th...,A container image is a standardized package th...
2,3,What is the difference between a container and...,A container is a runnable instance of an image.,A Docker image is a template that defines the ...,"A container is a running instance of an image,...",A container is an isolated process that runs f...


In [16]:
bilan_df.to_csv("results/bilan.csv",index=False)